In [0]:
import os, re
from pyspark.sql.functions import current_timestamp, col

catalog, schema = "workspace", "dm2_project"
base = f"/Volumes/{catalog}/{schema}"
landing, chk, ref = f"{base}/landing/sales", f"{base}/chk", f"{base}/reference"

# --- Source 1: STREAM (Auto Loader) -> bronze_sales ---
stream = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", f"{chk}/_schema/bronze_sales")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .load(landing)
    .withColumn("_ingest_ts", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path")))

# Delta dislikes spaces in column names: "Customer ID" -> "Customer_ID"
stream = stream.toDF(*[re.sub(r"[ ,;{}()\n\t=]", "_", c) for c in stream.columns])

(stream.writeStream.format("delta")
    .option("checkpointLocation", f"{chk}/bronze_sales")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"{catalog}.{schema}.bronze_sales")
    .awaitTermination())
print("bronze_sales updated")

# --- Source 2: OBJECT (batch file) -> bronze_country ---
country_file = [f for f in os.listdir(ref)
                if f.lower().endswith(".csv") and ("country" in f.lower() or "continent" in f.lower())][0]
country = spark.read.option("header", "true").csv(f"{ref}/{country_file}")
country.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.bronze_country")
print("bronze_country updated from:", country_file)

display(spark.table(f"{catalog}.{schema}.bronze_sales").limit(10))